In [ ]:
from __future__ import annotations
import os
import re
import time
import json
import pickle
import random
import argparse
from pathlib import Path
import json, urllib.parse, urllib.request
from typing import TypedDict, List, Dict, Any, Optional, Callable
from datetime import datetime

from dotenv import load_dotenv, find_dotenv 
from dataclasses import dataclass, asdict
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Optional dependencies - only for development/notebook use
try:
    from langchain_community.vectorstores import FAISS
    from langchain_community.document_loaders import TextLoader, DirectoryLoader
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False

try:
    from langchain_community.document_loaders import PyPDFLoader
    PDF_SUPPORT = True
except ImportError:
    PDF_SUPPORT = False

In [249]:
_ = load_dotenv(find_dotenv())

In [250]:
# State definition
class ArticleState(TypedDict, total=False):
    """State object tracking the article generation workflow."""
    topic: str
    outline: str
    research_sources: List[Dict[str, str]]
    article_draft: str
    validation_feedback: str
    is_valid: bool
    revision_count: int
    word_count: int
    
    # runtime article word count limits 
    target_words: int 
    tolerance: int 
    min_words: int 
    max_words: int 
    max_revisions: int 


In [251]:
class Config:
    """Central configuration"""
    MODEL_FALLBACK = "gemini-1.5-flash"
    
    # Paths
    PROMPTS_PATH = "./prompts"
    
    @classmethod
    def get_api_key(cls) -> str:
        key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
        if not key:
            raise ValueError("Set GEMINI_API_KEY or GOOGLE_API_KEY environment variable")
        return key

In [252]:
@dataclass 
class RuntimeConfig:
    model_name: str = "gemini-2.0-flash-exp"
    writer_temperature: float = 0.45 
    rewriter_temperature: float = 0.25 
    validator_temperature: float = 0.20

    # word limits 
    target_words: int = 1000 
    tolerance: int = 50
    max_revisions: int = 5

    # optional caps
    writer_max_tokens: int | None = 2048
    rewriter_max_tokens: int | None = 2048 
    validator_max_tokens: int | None = 2048 

def merge_runtime(overrides: dict | None) -> RuntimeConfig:
    base = RuntimeConfig()
    if overrides:
        for k, v in overrides.items():
            if hasattr(base, k):
                setattr(base, k, v)
    return base

In [253]:
# Prompt Manager
class PromptManager:
    """Manages prompts from text files - NEVER overwrites existing files"""
    
    DEFAULT_PROMPTS = {
        "planner.txt": """You are an expert content strategist creating article outlines.

Create a detailed outline for an article about: {topic}

The article should be approximately 1000 words. Include:
- Introduction with hook
- 3-4 main sections with subsections
- Conclusion with key takeaways

Output only the outline in clear, structured format.""",

        "researcher_queries.txt": """Extract 3-5 specific research queries from this topic and outline.

TOPIC: {topic}
OUTLINE: {outline}

Output only the queries, one per line.""",

        "researcher_insights.txt": """You are a research assistant. Provide comprehensive information for this query.

QUERY: {query}

Provide 3-4 key facts, insights, or perspectives. Include:
- Recent developments or trends
- Important statistics or data points
- Expert opinions or perspectives
- Relevant examples or case studies

Format each insight as a separate paragraph.""",

        "writer.txt": """You are an expert article writer. Write a comprehensive, engaging article.

TOPIC: {topic}

OUTLINE:
{outline}

RESEARCH INSIGHTS:
{research}

Requirements:
- Target: 950-1050 words
- Follow the outline structure
- Naturally incorporate research insights
- Engaging, professional style
- Clear introduction and conclusion
- Use specific examples and details

Write the complete article now:""",

        "validator.txt": """Review this article for quality.

ARTICLE:
{article}

Check:
1. Factual accuracy and consistency
2. Logical flow and coherence  
3. Grammar and style
4. Word count (target: 950-1050 words, current: {word_count})
5. Depth of content and examples

If acceptable, respond "VALID".
If issues exist, respond "REVISE: [specific issues]".

Be reasonably lenient - only reject for major problems.""",

        "rewriter.txt": """Revise this article based on feedback.

ORIGINAL ARTICLE:
{article}

FEEDBACK:
{feedback}

OUTLINE:
{outline}

Fix all identified issues while maintaining core content.
Ensure 950-1050 words.

Revised article:"""
    }
    
    def __init__(self, prompts_path: str = Config.PROMPTS_PATH):
        self.prompts_path = Path(prompts_path)
        self.prompts_path.mkdir(exist_ok=True)
        self._create_default_prompts()
    
    def _create_default_prompts(self):
        """Create default prompts ONLY if files don't exist"""
        created = 0
        for filename, content in self.DEFAULT_PROMPTS.items():
            filepath = self.prompts_path / filename
            if not filepath.exists():
                filepath.write_text(content.strip())
                created += 1
        
        if created > 0:
            print(f"✓ Created {created} default prompt files in {self.prompts_path}")
        else:
            print(f"✓ Using existing prompts from {self.prompts_path}")
    
    def get(self, prompt_name: str, **kwargs) -> str:
        """Load and format a prompt from file"""
        filepath = self.prompts_path / f"{prompt_name}.txt"
        if not filepath.exists():
            raise FileNotFoundError(f"Prompt not found: {filepath}")
        
        template = filepath.read_text()
        return template.format(**kwargs) if kwargs else template
    
    def save_version(self, prompt_name: str, suffix: str = None):
        """Save current prompt as versioned backup"""
        if suffix is None:
            suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        archive_dir = self.prompts_path / "archive"
        archive_dir.mkdir(exist_ok=True)
        
        source = self.prompts_path / f"{prompt_name}.txt"
        dest = archive_dir / f"{prompt_name}_{suffix}.txt"
        dest.write_text(source.read_text())
        print(f"✓ Archived: {dest.name}")

In [254]:
def enforce_word_limit(
    text, 
    min_words=950, 
    max_words=1050
):
    words = text.split()
    count = len(words)

    if count > max_words:
        # trim down while trying not to cut mid-sentence
        trimmed = " ".join(words[:max_words])
        if not trimmed.endswith(('.', '!', '?')):
            trimmed += "..."
        return trimmed

    elif count < min_words:
        # pad with a simple marker (better: re-prompt expansion)
        padding = " ".join(["[additional detail]" for _ in range(min_words - count)])
        return text + "\n\n" + padding

    return text

In [255]:
class GoogleSearcher:
    def __init__(self, google_api_key: str | None = None, google_cse_id: str | None = None, verbose: bool = True):
        self.enabled = False
        self.search_wrapper = None
        self.verbose = verbose

        api_key = google_api_key or os.getenv("GOOGLE_CUSTOM_SEARCH_KEY") or os.getenv("CDS_API_KEY")
        cse_id  = google_cse_id  or os.getenv("GOOGLE_CSE_ID")

        if not api_key:
            if verbose: print("ℹ️  Search disabled: missing Custom Search key (GOOGLE_CUSTOM_SEARCH_KEY/CDS_API_KEY).")
            return
        if not cse_id:
            if verbose: print("ℹ️  Search disabled: missing GOOGLE_CSE_ID (Programmable Search Engine ID).")
            return

        try:
            from langchain_google_community import GoogleSearchAPIWrapper
            # pass creds explicitly; don’t mutate env; don’t touch GEMINI key
            self.search_wrapper = GoogleSearchAPIWrapper(
                google_api_key=api_key,
                google_cse_id=cse_id,
            )
            self.enabled = True
            if verbose: print("✓ Google Search enabled")
        except Exception as e:
            if verbose: print(f"⚠️  Could not init GoogleSearchAPIWrapper: {e}")
            return

    def search(self, query: str, num_results: int = 5):
        if not self.enabled or not self.search_wrapper:
            return []
        try:
            raw = self.search_wrapper.results(query, num_results)
            return [{
                "title":   r.get("title") or "",
                "link":    r.get("link") or "",
                "content": r.get("snippet") or r.get("content") or "",
                "source":  f"Google: {r.get('title') or 'Unknown'}",
                "type":    "web_search"
            } for r in (raw or [])]
        except Exception as e:
            if self.verbose: print(f"⚠️  Google Search error: {e}")
            return []

In [256]:
searcher = GoogleSearcher(
    google_api_key=os.getenv("CDS_API_KEY"),
    google_cse_id=os.getenv("GOOGLE_CSE_ID"),
    verbose=True
)
print(searcher.search('"continuous integration"', num_results=2))

✓ Google Search enabled
⚠️  Google Search error: <HttpError 400 when requesting https://customsearch.googleapis.com/customsearch/v1?q=%22continuous+integration%22&cx=7618a9dd7f4e541e7&num=2&key=334866ba-9ee4-4311-bb36-6c32c50d7023&alt=json returned "API key not valid. Please pass a valid API key.". Details: "[{'message': 'API key not valid. Please pass a valid API key.', 'domain': 'global', 'reason': 'badRequest'}]">
[]


In [257]:
class WikipediaSearcher:
    def __init__(self, lang: str = "en", timeout: int = 15, ua: str = UA, verbose: bool = True):
        self.lang, self.timeout, self.ua, self.verbose = lang, timeout, ua, verbose
        self.hdrs = {"User-Agent": ua, "Accept": "application/json", "Accept-Language": "en-US,en;q=0.9", "Connection": "close", "Referer": "https://en.wikipedia.org/"}
        self.enabled = True

    def _get_json(self, url: str, params: Dict[str, Any] | None = None):
        if params:
            url = url + "?" + urllib.parse.urlencode(params)
        req = urllib.request.Request(url, headers=self.hdrs, method="GET")
        with urllib.request.urlopen(req, timeout=self.timeout) as resp:
            return json.loads(resp.read().decode("utf-8", errors="replace"))

    def search(self, query: str, num_results: int = 5):
        try:
            # Action API search + extracts
            base = f"https://{self.lang}.wikipedia.org/w/api.php"
            data = self._get_json(base, {"action":"query","list":"search","srsearch":query,"srlimit":max(1,num_results),"format":"json","utf8":1,"formatversion":2})
            hits = (data.get("query") or {}).get("search") or []
            if not hits: return []
            pageids = [str(h["pageid"]) for h in hits]
            detail = self._get_json(base, {"action":"query","prop":"extracts","explaintext":1,"exintro":1,"pageids":"|".join(pageids),"format":"json","utf8":1,"formatversion":2})
            pages = (detail.get("query") or {}).get("pages") or []
            out = []
            for p in pages[:num_results]:
                title = p.get("title","")
                extract = (p.get("extract") or "").strip()
                pid = p.get("pageid")
                link = f"https://{self.lang}.wikipedia.org/?curid={pid}" if pid else ""
                out.append({"title": title, "link": link, "content": extract, "engine": "wikipedia"})
            return out
        except Exception:
            # REST fallback: summary of first search hit
            try:
                base = f"https://{self.lang}.wikipedia.org/w/api.php"
                data = self._get_json(base, {"action":"query","list":"search","srsearch":query,"srlimit":1,"format":"json","utf8":1,"formatversion":2})
                hits = (data.get("query") or {}).get("search") or []
                if not hits: return []
                title = hits[0]["title"]
                safe = urllib.parse.quote(title.replace(" ", "_"))
                rest = f"https://{self.lang}.wikipedia.org/api/rest_v1/page/summary/{safe}"
                s = self._get_json(rest, None)
                return [{
                    "title": s.get("title", title),
                    "link": s.get("content_urls", {}).get("desktop", {}).get("page", ""),
                    "content": (s.get("extract") or "").strip(),
                    "engine": "wikipedia-rest"
                }]
            except Exception:
                return []

In [258]:
searcher = WikipediaSearcher()
queries = ["Continuous integration", "Machine learning", "CI/CD pipelines"]

for q in queries:
    print(f"\n🔎 Query: {q}")
    rows = searcher.search(q, num_results=3)
    if not rows:
        print("   ⚠️ No results")
    for i, r in enumerate(rows, 1):
        print(f"{i}. {r['title']}")
        print(f"   {r['link']}")
        print(f"   {r['content'][:200]}...\n")


🔎 Query: Continuous integration
1. Continuous integration
   https://en.wikipedia.org/?curid=1774081
   Continuous integration (CI) is the practice of integrating source code changes frequently and ensuring that the integrated codebase is in a workable state. Typically, developers merge changes to an in...

2. Comparison of continuous integration software
   https://en.wikipedia.org/?curid=22903426
   This is a compendium of software tools that support continuous integration....

3. Continuous deployment
   https://en.wikipedia.org/?curid=36120070
   Continuous deployment (CD) is a software engineering approach in which software functionalities are delivered frequently and through automated deployments.
Continuous deployment contrasts with continu...


🔎 Query: Machine learning
1. Neural network (machine learning)
   https://en.wikipedia.org/?curid=21523
   In machine learning, a neural network (also artificial neural network or neural net, abbreviated ANN or NN) is a computational mo

In [259]:
# Helper functions
def safe_llm_invoke(llm: ChatGoogleGenerativeAI, prompt: str, max_retries: int = 3) -> str:
    """Safely invoke LLM with retry logic"""
    for attempt in range(1, max_retries + 1):
        try:
            response = llm.invoke(prompt)
            return response.content
        except Exception as e:
            error_text = str(e).lower()
            print(f"⚠️  Attempt {attempt}/{max_retries} failed: {e}")
            
            if "429" in error_text or "quota" in error_text:
                wait_time = min(10 * attempt, 60)
                print(f"🚦 Rate limit - waiting {wait_time}s...")
                time.sleep(wait_time)
                continue
            
            if "timeout" in error_text or "connection" in error_text:
                time.sleep(5 * attempt)
                continue
            
            if attempt < max_retries:
                time.sleep(5 * attempt)
                continue
            
            raise
        
    return "(LLM invocation failed after retries)"

In [260]:
# Agent nodes
def planner_node(
    state: ArticleState, 
    llm: ChatGoogleGenerativeAI, 
    pm: PromptManager
) -> ArticleState:
    """Creates article outline"""
    print(f"\n📋 PLANNER: Creating outline for '{state['topic']}'")
    
    prompt = pm.get(
        "planner", 
        topic=state["topic"],
        word_count=state["word_count"]
    )
    outline = safe_llm_invoke(llm, prompt)
    
    state["outline"] = outline
    print(f"✓ Outline created ({len(outline.split())} words)")
    return state


def researcher_node(
    state: ArticleState, 
    llm: ChatGoogleGenerativeAI,
    pm: PromptManager,
    searcher: Optional[DuckDuckGoSearcher] = None 
) -> ArticleState:
    """Generates research insights using LLM"""
    print(f"\n🔍 RESEARCHER: Gathering insights")
    
    # Extract research queries
    query_prompt = pm.get("researcher_queries", 
                          topic=state["topic"], 
                          outline=state["outline"][:500])
    query_response = safe_llm_invoke(llm, query_prompt)
    queries = [q.strip() for q in query_response.split('\n') if q.strip()][:5]
    
    insights: List[str] = []
    sources_all: List[Dict[str, str]] = []
    
    # Generate insights for each query
    for i, query in enumerate(queries, 1):
        print(f"  Researching: {query[:60]}...")

        results = []

        if searcher and searcher.enabled:
            results = searcher.search(query, num_results=3)
        else:
            # No web search results, LLM will use prior knowledge
            results = []
        
        print("Results: ", results)

        # Record sources 
        for r in results:
            sources_all.append({
                "title": r.get("title", ""),
                "link": r.get("link", ""),
                "content": r.get("content", ""),
                "query": query
            })

        if results:
            evidence_lines = []
            for r in results:
                title = r.get("title", "") or r.get("link", "")
                snippet = (r.get("content", "") or "")[:240]
                link = r.get("link", "")
                evidence_lines.append(f"- {title} — {snippet} [{link}]")
            evidence = "\n".join(evidence_lines)
        else:
            evidence = "(No external sources available; use domain knowledge.)"

        # Ask LLM to synthesize a succinct, sourced insight
        insight_prompt = pm.get("researcher_insights", query=query, evidence=evidence)
        insight = safe_llm_invoke(llm, insight_prompt)
        insights.append(f"Query {i}: {query}\n\n{insight}")
    
    state["research_sources"] = sources_all
    print(f"✓ Generated {len(insights)} research insights")
    
    return state

def writer_node(
    state: ArticleState, 
    llm: ChatGoogleGenerativeAI,
    pm: PromptManager,
    searcher
) -> ArticleState:
    """Writes or revises article"""
    print(f"\n✍️  WRITER: Composing article (revision {state['revision_count'] + 1})")
    
    sources = searcher.search(state["topic"], num_results=3) if searcher else []
    state["research_sources"] = sources
    
    evidence = "\n".join(
        f"- {s['title']} — {s['content'][:200]} [{s['link']}]"
        for s in sources if s.get("link")
    )
    sources_md = "\n".join(
        f"- [{s['title']}]({s['link']})"
        for s in sources if s.get("link")
    )

    prompt = pm.get(
        "writer",
        topic=state["topic"],
        outline=state["outline"],
        min_words=state.get("min_words"),
        max_words=state.get("max_words"),
        sources=sources_md,
        evidence=evidence
    )
    
    article = safe_llm_invoke(llm, prompt)
    
    state["article_draft"] = article
    state["revision_count"] += 1
    state["word_count"] = len(article.split())
    
    print(f"✓ Article written ({state['word_count']} words)")
    return state


def validator_node(
    state: ArticleState, 
    llm: ChatGoogleGenerativeAI,
    pm: PromptManager
) -> ArticleState:
    """Validates article quality and word count"""
    print(f"\n✅ VALIDATOR: Checking quality")
    
    word_count = len(state["article_draft"].split())
    min_words = state.get("min_words")
    max_words = state.get("max_words")
    
    feedback_parts = []
    
    # Word count check
    if word_count < min_words:
        feedback_parts.append(f"Too short: {word_count} words (need {min_words}-{max_words})")
    elif word_count > max_words:
        feedback_parts.append(f"Too long: {word_count} words (need {min_words}-{max_words})")
    
    # Quality check (if word count ok or last revision)
    if not feedback_parts or state["revision_count"] >= state.get("max_revisions"):
        prompt = pm.get(
            "validator",
            article=state["article_draft"],
            word_count=word_count,
            min_words=state["min_words"],
            max_words=state["max_words"]
        )

        quality_feedback = safe_llm_invoke(llm, prompt, max_retries=2)
        
        if "VALID" not in quality_feedback.upper():
            feedback_parts.append(quality_feedback)
    
    # Set validation result
    state["is_valid"] = len(feedback_parts) == 0
    state["validation_feedback"] = "\n\n".join(feedback_parts)
    
    if state["is_valid"]:
        print("✓ Article validated")
    else:
        print(f"⚠️  Issues found")
    
    return state

def rewriter_node(
    state: ArticleState, 
    llm: ChatGoogleGenerativeAI,
    pm: PromptManager
) -> ArticleState:
    """Revises article based on feedback"""
    print(f"\n🔄 REWRITER: Revising (attempt {state['revision_count']})")
    
    prompt = pm.get(
        "rewriter",
        article=state["article_draft"],
        feedback=state["validation_feedback"],
        outline=state["outline"],
        min_words=state.get("min_words"),
        max_words=state.get("max_words")
    )
    
    revised = safe_llm_invoke(llm, prompt)
    
    state["article_draft"] = revised
    state["word_count"] = len(revised.split())

    state["revision_count"] += 1
    
    print(f"✓ Article revised ({state['word_count']} words)")
    return state

In [261]:
# Routing 
def route_validator(state: ArticleState) -> str:
    """Route based on validation result"""
    max_revs = state.get("max_revisions", 5)

    if state["is_valid"]:
        return "end"
    elif state["revision_count"] >= max_revs:
        print(f"\n⚠️  Max revisions reached. Using current version.")
        return "end"
    else:
        return "revise"

In [262]:
# Workflow builder 
def create_workflow(
    config: Config,
    runtime_config: RuntimeConfig,
    searcher
) -> StateGraph:
    """Create the LangGraph workflow"""
    prompt_manager = PromptManager(prompts_path=config.PROMPTS_PATH)
    
    # Initialize components
    llm_writer = ChatGoogleGenerativeAI(
        model=runtime_config.model_name,
        temperature=runtime_config.writer_temperature,
        google_api_key=Config.get_api_key(),
        max_tokens=runtime_config.writer_max_tokens
    )

    llm_rewriter = ChatGoogleGenerativeAI(
        model=runtime_config.model_name,
        temperature=runtime_config.rewriter_temperature,
        google_api_key=Config.get_api_key(),
        max_tokens=runtime_config.rewriter_max_tokens
    )

    llm_validator = ChatGoogleGenerativeAI(
        model=runtime_config.model_name,
        temperature=runtime_config.validator_temperature,
        google_api_key=Config.get_api_key(),
        max_tokens=runtime_config.validator_max_tokens
    )

    # Create graph
    workflow = StateGraph(ArticleState)
    
    # Add nodes
    workflow.add_node("planner", 
                     lambda s: planner_node(s, llm_writer, prompt_manager))
    workflow.add_node("writer",
                     lambda s: writer_node(s, llm_writer, prompt_manager, searcher))
    workflow.add_node("validator",
                     lambda s: validator_node(s, llm_validator, prompt_manager))
    workflow.add_node("rewriter",
                     lambda s: rewriter_node(s, llm_rewriter, prompt_manager))
    
    # Define flow
    workflow.set_entry_point("planner")
    workflow.add_edge("planner", "writer")
    workflow.add_edge("writer", "validator")
    
    # Conditional routing
    workflow.add_conditional_edges(
        "validator",
        route_validator,
        {"revise": "rewriter", "end": END}
    )
    workflow.add_edge("rewriter", "validator")
    
    return workflow.compile()# Workflow builder 

In [263]:
_SENT_SPLIT = re.compile(r'(?<=[.!?])\s+')

def _word_count(s: str) -> int:
    return len(s.split())

def sentence_reduce_random(
    text: str,
    max_words: int,
    *,
    preserve_first_last: bool = True,
    seed: Optional[int] = 42
) -> Tuple[str, int]:
    """
    Randomly remove full sentences until total word count <= max_words.
    Preserves markdown structure lines and code blocks.
    Optionally preserves the very first and very last sentence of prose.
    
    Returns (reduced_text, reduced_word_count).
    """
    if _word_count(text) <= max_words:
        return text, _word_count(text)

    rng = random.Random(seed)

    # Parse into lines, track code blocks
    lines = text.splitlines()
    in_code = False

    # For each line, either keep as-is (markdown structure/code)
    # or split into sentences we can remove.
    line_sentences: List[Optional[List[str]]] = []  # None => structural line; list => sentences
    removable_indices: List[Tuple[int, int]] = []   # (line_idx, sent_idx) candidates to remove
    global_sent_positions: List[Tuple[int, int]] = []  # to know first/last sentence

    structural_prefixes = tuple(["#", ">", "-", "*", "+"])  # headers, quotes, lists
    numeric_bullet = re.compile(r"^\s*\d+\.\s+")

    # Build sentence lists and candidate removals
    for li, raw in enumerate(lines):
        line = raw.rstrip()

        # code block fences
        if line.strip().startswith("```"):
            in_code = not in_code
            line_sentences.append(None)
            continue

        # structural or in code => not reducible
        if in_code or line.strip().startswith(structural_prefixes) or numeric_bullet.match(line):
            line_sentences.append(None)
            continue

        if not line.strip():
            line_sentences.append(None)
            continue

        # Split prose into sentences
        sents = _SENT_SPLIT.split(line.strip())
        line_sentences.append(sents)

    # Build global sentence index for preserve_first_last
    for li, sents in enumerate(line_sentences):
        if sents is None:
            continue
        for si, _ in enumerate(sents):
            global_sent_positions.append((li, si))

    # Candidates to remove = all sentences except preserved ones
    first_guard = global_sent_positions[0] if preserve_first_last and global_sent_positions else None
    last_guard  = global_sent_positions[-1] if preserve_first_last and len(global_sent_positions) > 1 else None

    for li, sents in enumerate(line_sentences):
        if sents is None:
            continue
        for si, _ in enumerate(sents):
            if first_guard and (li, si) == first_guard:
                continue
            if last_guard and (li, si) == last_guard:
                continue
            removable_indices.append((li, si))

    # Shuffle removal order
    rng.shuffle(removable_indices)

    # Current total words
    def current_word_count() -> int:
        rebuilt_lines = []
        for sents in line_sentences:
            if sents is None:
                # structural or code lines remain as original text
                # We'll reconstruct from original lines for accuracy
                continue
        # Faster: compute from current reconstruction below
        return _word_count(reconstruct_text())

    def reconstruct_text() -> str:
        out = []
        for li, sents in enumerate(line_sentences):
            if sents is None:
                out.append(lines[li])
            else:
                joined = " ".join(s for s in sents if s is not None)
                out.append(joined)
        return "\n".join(out).strip()

    total_words = _word_count(reconstruct_text())
    if total_words <= max_words:
        return reconstruct_text(), total_words

    # Remove sentences randomly until <= max_words (skipping already-removed)
    for li, si in removable_indices:
        sents = line_sentences[li]
        if sents is None:
            continue
        if 0 <= si < len(sents) and sents[si] is not None:
            sents[si] = None  # mark removed
            total_words = _word_count(reconstruct_text())
            if total_words <= max_words:
                reduced = reconstruct_text()
                return reduced, total_words

    # If we still didn't get under (rare), fall back to a hard cap at word boundary
    reduced = reconstruct_text()
    words = reduced.split()
    if len(words) > max_words:
        reduced = " ".join(words[:max_words])
    return reduced, _word_count(reduced)

In [264]:
class ArticleSystem:
    """
    Main interface for article generation system
    
    Usage:
        system = ArticleSystem()
        system.setup()
        result = system.generate("Your topic")
        
        # Save article
        with open("article.txt", "w") as f:
            f.write(result["article"])
    """
    
    def __init__(self):
        """Initialize system"""
        self.prompt_manager = None
        self.workflow = None
        self._is_ready = False
    
    def setup(
        self, 
        searcher,
        runtime_overrides: dict | None = None,
        config: Config | None = None
    ):
        """Setup system - creates prompts and workflow"""
        print("🔧 Setting up Article Generation System...")

        self.config = config or Config()
        
        # Initialize prompts
        self.prompt_manager = PromptManager()

        self.runtime_config = merge_runtime(runtime_overrides)
        
        # Create workflow
        self.workflow = create_workflow(
            config=self.config,
            runtime_config=self.runtime_config,
            searcher=searcher
        )
        self._is_ready = True
        
        print("\n✅ Setup complete!")
        print(f"   - Prompts: {self.config.PROMPTS_PATH}")
        print(f"   - Model:   {self.runtime_config.model_name}")
        print(f"   - Target:  {self.runtime_config.target_words} ± {self.runtime_config.tolerance}")
        print(f"   - Max revs:{self.runtime_config.max_revisions}")
        print(f"   - Ready to generate!")
    
    def generate(
        self,
        topic: str,
        runtime_overrides: dict | None = None
    ) -> Dict[str, Any]:
        """
        Generate article
        
        Args:
            topic: Article topic
            
        Returns:
            Dictionary with article and metadata
        """
        if not self._is_ready or runtime_overrides:
            self.setup(runtime_overrides)

        min_words = self.runtime_config.target_words - self.runtime_config.tolerance
        max_words = self.runtime_config.target_words + self.runtime_config.tolerance 
        
        print(f"\n{'='*70}")
        print(f"🚀 Generating Article: {topic}")
        print(f"{'='*70}")
        
        # Initialize state
        initial_state = ArticleState(
            topic=topic,
            outline="",
            article_draft="",
            validation_feedback="",
            is_valid=False,
            revision_count=0,
            word_count=0,

            target_words=self.runtime_config.target_words,
            tolerance=self.runtime_config.tolerance,
            min_words=min_words,
            max_words=max_words,
            max_revisions=self.runtime_config.max_revisions,
        )
        
        # Run workflow
        final_state = self.workflow.invoke(initial_state)
        
        print(f"\n{'='*70}")
        print(f"✨ Generation Complete!")
        print(f"   - Word count: {final_state['word_count']}")
        print(f"   - Revisions: {final_state['revision_count']}")
        print(f"{'='*70}\n")
        
        # Safeguarded version (trim by sentence to avoid cutting flow)
        def _sentence_safe_enforce(text: str, mn: int, mx: int) -> str:
            try:
                import re
                words = text.split()
                if len(words) <= mx:
                    return text
                sentences = re.split(r'(?<=[.!?])\s+', text)
                out, total = [], 0
                for s in sentences:
                    w = len(s.split())
                    if total + w > mx:
                        break
                    out.append(s)
                    total += w
                return " ".join(out)
            except Exception:
                # Fallback: hard cap (last resort)
                return " ".join(text.split()[:mx])

        def sentence_cap_enforce(text: str, max_words: int) -> str:
            """
            Truncate text so it never exceeds max_words,
            but only cut at sentence boundaries.
            """
            words = text.split()
            if len(words) <= max_words:
                return text

            sentences = re.split(r'(?<=[.!?])\s+', text)
            out, total = [], 0
            for s in sentences:
                w = len(s.split())
                if total + w > max_words:
                    break
                out.append(s)
                total += w

            return " ".join(out)

        safeguarded_article, safeguarded_wc  = sentence_reduce_random(
            final_state["article_draft"],
            final_state["max_words"],
            preserve_first_last=True,
            seed=42
        )
        met_limits_without_safeguard = (
            final_state["word_count"] >= final_state["min_words"]
            and final_state["word_count"] <= final_state["max_words"]
        )

        # Build UI payload
        return {
            # 1) Final article from the LLM after max revisions
            "final_article_llm": final_state["article_draft"],

            # 2) If it was valid or not
            "was_valid": final_state["is_valid"],

            # 3) Other details
            "details": {
                "topic": final_state["topic"],
                "outline": final_state["outline"],
                "sources": final_state["research_sources"],
                "word_count": final_state["word_count"],
                "validation_feedback": final_state["validation_feedback"],
                "revisions_used": final_state["revision_count"],
                "limits": {
                    "target_words": final_state["target_words"],
                    "tolerance": final_state["tolerance"],
                    "min_words": final_state["min_words"],
                    "max_words": final_state["max_words"],
                },
                # runtime model settings exposed from self.runtime (not in state)
                "model": self.runtime_config.model_name,
                "temperatures": {
                    "writer": self.runtime_config.writer_temperature,
                    "rewriter": self.runtime_config.rewriter_temperature,
                    "validator": self.runtime_config.validator_temperature,
                },
                "max_tokens": {
                    "writer": self.runtime_config.writer_max_tokens,
                    "rewriter": self.runtime_config.rewriter_max_tokens,
                    "validator": self.runtime_config.validator_max_tokens,
                },
            },

            # 4) Max revisions
            "max_revisions": self.runtime_config.max_revisions,

            # 5) Revised article that meets the word count limit after applying safeguard
            "safeguarded_article": safeguarded_article,
            "safeguarded_word_count": safeguarded_wc,

            # convenience flag
            "met_limits_without_safeguard": met_limits_without_safeguard,
        }
    
    def get_stats(self) -> Dict[str, Any]:
        """Get system statistics"""
        return {
            "ready": self._is_ready,
            "prompts_path": str(Path(Config.PROMPTS_PATH).absolute()),
            "config": {
                "model": self.runtime_config.model_name if self.runtime_config else None,
                "writer_temperature": getattr(self.runtime_config, "writer_temperature", None),
                "rewriter_temperature": getattr(self.runtime_config, "rewriter_temperature", None),
                "validator_temperature": getattr(self.runtime_config, "validator_temperature", None),
                "target_words": getattr(self.runtime_config, "target_words", None),
                "tolerance": getattr(self.runtime_config, "tolerance", None),
                "max_revisions": getattr(self.runtime_config, "max_revisions", None),
                "writer_max_tokens": getattr(self.runtime_config, "writer_max_tokens", None),
                "rewriter_max_tokens": getattr(self.runtime_config, "rewriter_max_tokens", None),
                "validator_max_tokens": getattr(self.runtime_config, "validator_max_tokens", None),
            }
        } 

In [272]:
# Convenience functions
def quick_setup() -> ArticleSystem:
    """Quick setup for interactive use"""
    system = ArticleSystem()
    system.setup()
    return system

def generate_and_save(
    topic: str, 
    filename: str = None
) -> str:
    """Generate article and save to file"""
    system = quick_setup()
    result = system.generate(topic)
    
    if filename is None:
        filename = f"article_{topic.replace(' ', '_')[:40]}.txt"
    
    with open(filename, "w") as f:
        f.write(result["article"])
    
    print(f"💾 Saved to {filename}")
    return result["article"]

In [281]:
def run_article_writer(
    topic: str = "CI/CD Pipelines",
    target_words: int = 1000,
    tolerance: int = 50,
    max_revisions: int = 3,
    model_name: str = "gemini-2.0-flash-exp",
    searcher = WikipediaSearcher(),
    writer_temperature: float = 0.2,
    rewriter_temperature: float = 0.1,
    validator_temperature: float = 0.0,
    writer_max_tokens: int | None = None,
    rewriter_max_tokens: int | None = None,
    validator_max_tokens: int | None = None,
    save_files: bool = False
):
    print("="*70); print("ARTICLE GENERATION SYSTEM"); print("="*70)

    runtime_overrides = {
        "model_name": model_name,
        "writer_temperature": writer_temperature,
        "rewriter_temperature": rewriter_temperature,
        "validator_temperature": validator_temperature,
        "writer_max_tokens": writer_max_tokens,
        "rewriter_max_tokens": rewriter_max_tokens,
        "validator_max_tokens": validator_max_tokens,
        "target_words": target_words,
        "tolerance": tolerance,
        "max_revisions": max_revisions,
    }

    system = ArticleSystem()
    system.setup(
        config=Config(),
        runtime_overrides=runtime_overrides,
        searcher=searcher
    )

    topic = (topic or "The Future of Artificial Intelligence").strip() or "The Future of Artificial Intelligence"
    result = system.generate(topic)

    if save_files:
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_dir = Path(".")
        llm_path = out_dir / f"article_llm_{stamp}.md"
        safe_path = out_dir / f"article_safe_{stamp}.md"
        meta_path = out_dir / f"article_meta_{stamp}.json"

        llm_path.write_text(result["final_article_llm"], encoding="utf-8")
        safe_path.write_text(result["safeguarded_article"], encoding="utf-8")
    else:
        llm_path, safe_path, meta_path = None, None, None

    meta = {
        "was_valid": result["was_valid"],
        "max_revisions": result["max_revisions"],
        "met_limits_without_safeguard": result["met_limits_without_safeguard"],
        "details": result["details"],
        "files": {"final_article_llm": llm_path, "safeguarded_article": llm_path},
    }

    return {
        "llm_path": str(llm_path), 
        "safe_path": str(safe_path), 
        "meta_path": str(meta_path), 
        "result": result
    }

In [282]:
response = run_article_writer(
    topic="CI/CD Pipelines",
    target_words=1000,
    tolerance=50,
    max_revisions=3
)

ARTICLE GENERATION SYSTEM
🔧 Setting up Article Generation System...
✓ Using existing prompts from prompts
✓ Using existing prompts from prompts

✅ Setup complete!
   - Prompts: ./prompts
   - Model:   gemini-2.0-flash-exp
   - Target:  1000 ± 50
   - Max revs:3
   - Ready to generate!

🚀 Generating Article: CI/CD Pipelines

📋 PLANNER: Creating outline for 'CI/CD Pipelines'
✓ Outline created (95 words)

✍️  WRITER: Composing article (revision 1)
✓ Article written (1131 words)

✅ VALIDATOR: Checking quality
⚠️  Issues found

🔄 REWRITER: Revising (attempt 1)
✓ Article revised (1291 words)

✅ VALIDATOR: Checking quality
⚠️  Issues found

🔄 REWRITER: Revising (attempt 2)
✓ Article revised (1339 words)

✅ VALIDATOR: Checking quality
⚠️  Issues found

⚠️  Max revisions reached. Using current version.

✨ Generation Complete!
   - Word count: 1339
   - Revisions: 3

